# Cheminformatics Eval — execute agent, **with vs without skills**

This notebook evaluate how much the **cheminformatics** skill helps the ChemSafe's **execute agent** answers
cheminformatics questions

## Method
Given the same set of questions, the **execute agennt** attempts to answer those with and without **cheminformatics** skill. 

Both conditions can still call libraries like `rdkit`, `admet_ai`, `deepchem`, and `pubchempy` from Python; 
the only thing vary is the incorporation of **cheminforatics skill**. So the difference in scores isolates the value of the skills.

### LLM-as-judge scorer

Each agent answer is free-form text. A second model grades it and returns
`correct` / `partial` / `incorrect`, matching numbers within **±2% relative**
tolerance (±1e-3 near zero), categorical labels (GHS class, H-codes,
active/inactive, …) by meaning, SMILES by chemical equivalence, and prose by
whether the key claims are present.

### How this notebook is organized

* **Sections 1–12 build the toolkit** — each defines one piece and explains it.
* **Sections 13–17 run it** — build the agents, smoke-test, (optionally) run the full
  set, then aggregate and compare.

## 1. Import dependencies

In [ ]:
import csv
import glob
import json
import os
import random
import re
import sys
import time
import traceback
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Literal, Optional, Union
import pandas as pd  


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` until we find the project, and return that folder."""
    for parent in [start, *start.parents]:
        if (parent / "core" / "agents" / "execute_agent.py").exists():
            return parent
    raise RuntimeError(
        "Could not find the repo root. Open this notebook from inside the "
        "chemsafe-agent repository."
    )


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# This folder holds the notebook, the dataset CSV, and the results we write.
EVAL_DIR = REPO_ROOT / "analysis" / "cheminformatics_eval"
RESULTS_DIR = EVAL_DIR / "eval_runs"           # where run outputs are stored
RESULTS_PATH = RESULTS_DIR / "results.jsonl"   # main append-only results file

print("repo root:", REPO_ROOT)
print("eval dir :", EVAL_DIR)

In [ ]:
from langchain.chat_models import init_chat_model           
from langchain_core.messages import AIMessage, HumanMessage  
from langgraph.prebuilt import create_react_agent            
from pydantic import BaseModel, Field                        


from app.config import EXECUTE_MODEL, OPENAI_API_KEY, SUMMARY_MODEL  
from core.agents.execute_agent import build_execute_agent            
from core.agents.context import build_uncompressed_pre_model_state   
from core.prompts.prompts import EXECUTE_AGENT_FREE_SYSTEM_PROMPT, EXECUTE_SKILLS_BLOCK
from backend.utils.skills_format import format_skill_summaries

from core.tools.python_executor import python_executor, reset_python_state  
from backend.utils.output_paths import (                           
    set_current_conversation_id,
    set_current_user_id,
)

## 3. Pin an output / read scope

The agent's `python_executor` writes any files it creates into a **scoped** directory,
and `read_files` is likewise restricted to a scope. The scope is taken from two
context variables (a user id and a conversation id).

We pin them to a fixed "eval" identity so anything the agent writes lands under
`persistence/results/cheminf-eval/...` and never touches a real user's space.
`set_eval_scope()` is safe to call as many times as you like.

In [ ]:
EVAL_USER_ID = "cheminf-eval"
EVAL_CONVERSATION_ID = "cheminf-eval-run"


def set_eval_scope(user_id: str = EVAL_USER_ID, conversation_id: str = EVAL_CONVERSATION_ID):
    """Pin the output/read scope for the eval session (idempotent)."""
    set_current_user_id(user_id)
    set_current_conversation_id(conversation_id)

## 4. Build the WITH-skills agent

This builds the **execute agent** with **cheminformatics** skill

In [ ]:
# Skills this eval agent is allowed to know about (instead of the full EXECUTE_SKILLS).
EVAL_SKILLS = ["cheminformatics", "database_traversal"]
EVAL_SKILLS_BLOCK = format_skill_summaries(EVAL_SKILLS)
# Only activate two skills rather than full skill set of execution agent
EVAL_EXECUTE_SYSTEM_PROMPT = EXECUTE_AGENT_FREE_SYSTEM_PROMPT.replace(
    EXECUTE_SKILLS_BLOCK, EVAL_SKILLS_BLOCK
)


def build_eval_agent(model: Optional[str] = None):
    """Build the execute agent restricted to the EVAL_SKILLS (cheminformatics +
    database_traversal).
    """
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is not set. Add it to the repo-root .env.")
    llm = init_chat_model(model or EXECUTE_MODEL, model_provider="openai", api_key=OPENAI_API_KEY)
    prompt, name = EVAL_EXECUTE_SYSTEM_PROMPT, "execute_agent_free"
    return build_execute_agent(
        llm, pre_model_hook=build_uncompressed_pre_model_state, name=name, prompt=prompt
    )

## 5. Build the NO-skills baseline agent

This builds the **execute agent** without **cheminformatics** skill

In [ ]:
NO_SKILLS_SYSTEM_PROMPT = """
You are the execution agent for chemical safety-relevant workflows. You handle short, well-scoped requests directly, without an external plan. Your job is to reach a correct, evidence-grounded answer efficiently and use tools whenever they produce real progress.

---

## Primary Role

You operate without a pre-approved plan. You decide the working approach yourself, but you **must**:

1. Keep the user's request as the source of truth.
2. Use tools to produce real evidence (executions, validations, and computed results).
3. Stay focused — do only the work the request actually requires; do not invent extra phases.
4. Surface any uncertainty or limitation explicitly instead of hiding it.

You **must not**:

- Re-plan the task into a heavyweight multi-stage workflow when the request is small.
- Skip tool use and answer purely from reasoning when computation or execution is needed.
- Ask the user what to do next unless execution is truly blocked.

---

## Execution Posture

The expected runtime pattern is:

1. Restate the request in one short line and identify what evidence the answer needs.
2. Load only the context required.
3. Execute with one or more focused tool calls.
4. Inspect results and either finish or recover from errors.
5. Produce the final answer grounded in observed evidence.

Prefer small, focused `python_executor` probes. Reuse Python state when useful; `reset_python_state` when it gets stale.

---

## Tool Discipline

- `python_executor` performs inspection, lightweight analysis, validation, and file generation.
- `reset_python_state` is a recovery tool for contaminated Python state.

**Rules:**

- Do not declare the task complete from reasoning alone when tool evidence is available.
- Recovery is part of execution — adapt the approach on failure rather than abandoning.
- Use the injected output helpers (`prepare_output_path`, `ensure_output_dir`) for any generated files.

---

## Safety Guardrails

1. For any safety-relevant action, threshold, or recommendation, ground it in computed or cited evidence before finalizing the answer.
2. For any data-dependent claim, inspect the data before summarizing it.
3. Never invent tool outputs, values, thresholds, or results.
4. If you cannot complete the request safely, say so explicitly and surface the blocker.
"""

def build_eval_agent_no_skills(model: Optional[str] = None):
    """Baseline agent: python_executor only (NO read_files), skill-free prompt."""
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is not set. Add it to the repo-root .env.")
    llm = init_chat_model(model or EXECUTE_MODEL, model_provider="openai", api_key=OPENAI_API_KEY)
    return create_react_agent(
        model=llm,
        tools=[python_executor, reset_python_state],  # no read_files -> no skill to read
        name="execute_agent_no_skills",
        prompt=NO_SKILLS_SYSTEM_PROMPT,
        pre_model_hook=build_uncompressed_pre_model_state,
    )

## 6. Load the evaluation dataset

Loads the CSV into a list of plain dictionaries (one per question). Two cleaning
steps matter:

* The CSV's column headers are wrapped in single quotes, so `_clean_key` strips them.
* For `dict` / `scalar` rows the reference answer also ships as JSON in the
  `ref_answer_parsed` column; `_safe_json` parses it into `row['_ref_parsed']` (shown to
  the judge as the structured reference). `prose` rows have none.

`stratified_sample` picks a few rows per domain — handy for a quick, cheap signal
before committing to the full set.

In [ ]:
def _clean_key(key: str) -> str:
    """Strip surrounding quotes/spaces from a CSV column name."""
    return (key or "").strip().strip("'\"").strip()


def _safe_json(text: str):
    """Parse a JSON string, or return None if it is empty/invalid."""
    text = (text or "").strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        return None


def load_dataset(path: Optional[Union[str, Path]] = None) -> list[dict]:
    """Load the eval CSV into normalized row dicts.

    Adds row['_ref_parsed']: the parsed JSON reference for dict/scalar rows.
    """
    p = Path(path)
    with open(p, newline="", encoding="utf-8") as fh:
        raw_rows = list(csv.DictReader(fh))
    rows: list[dict] = []
    for raw in raw_rows:
        row = {_clean_key(k): (v if v is not None else "") for k, v in raw.items()}
        row["_ref_parsed"] = _safe_json(row.get("ref_answer_parsed", ""))
        rows.append(row)
    return rows


def stratified_sample(rows: list[dict], per_domain: int = 2, seed: int = 0) -> list[dict]:
    """Pick `per_domain` rows from each domain (a cheap, balanced subset)."""
    rnd = random.Random(seed)
    by_domain: dict[str, list[dict]] = defaultdict(list)
    for row in rows:
        by_domain[row.get("domain", "?")].append(row)
    out: list[dict] = []
    for _domain, domain_rows in sorted(by_domain.items()):
        pool = domain_rows[:]
        rnd.shuffle(pool)
        out.extend(pool[:per_domain])
    return out

## 7. Run the agent on one question (and capture a trace)

`run_agent_on_query` sends one question to an agent and collects the result. The agent returns a list of chat messages:

* `_text` flattens a message's content to plain text.
* `_final_answer` picks the **last assistant message that is real text** (not a tool
  call) — that's the agent's answer.
* `_summarize_trace` inspects the messages to record **how** the agent worked: which
  tools it called, how many Python runs, which `SKILL.md` files it opened, and whether
  its code imported `core.skills`. (This is how we can later confirm the no-skills
  agent really used no skills.)

Any error is caught and returned as `ok=False`, so a single bad row never aborts a long
run.

In [ ]:
def _text(content: Any) -> str:
    """Flatten a message's content (str, or list of parts) into plain text."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
            elif isinstance(item, str):
                parts.append(item)
        return "\n".join(parts).strip()
    return str(content).strip()


def _final_answer(messages: list) -> str:
    """Last assistant message with text content and no pending tool calls."""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and not getattr(msg, "tool_calls", None):
            text = _text(msg.content)
            if text:
                return text
    for msg in reversed(messages):  # fallback
        if isinstance(msg, AIMessage):
            text = _text(msg.content)
            if text:
                return text
    return ""


def _summarize_trace(messages: list) -> dict:
    """Record how the agent worked: tools used, SKILL.md files read, core.skills imports."""
    tools_used: list[str] = []
    skills_read: set[str] = set()
    skills_in_code: set[str] = set()
    n_python_runs = 0
    for msg in messages:
        for call in getattr(msg, "tool_calls", None) or []:
            name = call.get("name", "")
            args = call.get("args", {}) or {}
            tools_used.append(name)
            if name == "read_files":
                fp = str(args.get("file_path", ""))
                if "skills/" in fp:
                    skills_read.add(fp.split("skills/")[1].split("/")[0])
            elif name == "python_executor":
                n_python_runs += 1
                for mod in re.findall(r"core\.skills\.(\w+)", str(args.get("code", ""))):
                    skills_in_code.add(mod)
    return {
        "tool_counts": dict(Counter(tools_used)),
        "n_python_runs": n_python_runs,
        "skills_read": sorted(skills_read),
        "skills_used_in_code": sorted(skills_in_code),
    }


def run_agent_on_query(agent, question: str, *, recursion_limit: int = 50) -> dict:
    """Invoke the agent on one question; return answer + trace + timing.

    recursion_limit: max agent steps (the agent may take many tool turns).
    """
    started = time.time()
    try:
        result = agent.invoke(
            {"messages": [HumanMessage(content=question)]},
            config={"recursion_limit": recursion_limit},
        )
        messages = result["messages"]
        return {
            "ok": True,
            "answer": _final_answer(messages),
            "trace": _summarize_trace(messages),
            "n_messages": len(messages),
            "elapsed_s": round(time.time() - started, 2),
        }
    except Exception as exc:  # keep the loop alive; record the failure
        return {
            "ok": False,
            "answer": "",
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(),
            "elapsed_s": round(time.time() - started, 2),
        }

## 8. The LLM judge — the verdict

The verdict comes from a second LLM acting as a grader.

* `JudgeVerdict` is the **Pydantic model** that is forced the judge to return
  (`correct`/`partial`/`incorrect` plus supporting detail). Using structured output
  means we get clean fields back, not free text to parse.
* `JUDGE_SYSTEM` is the rubric: numbers within tolerance, categorical labels matched by
  meaning, SMILES by chemical equivalence, prose by claim coverage, and "don't penalize
  extra correct content".
* `build_judge` wires up the model with structured output.
* `judge_answer` hands the judge the question, the reference (and its parsed form), and
  the agent's answer, and returns the verdict as a dict.

In [ ]:
class JudgeVerdict(BaseModel):
    """Structured verdict the judge must return."""

    verdict: Literal["correct", "partial", "incorrect"] = Field(
        description="correct = all required content right; partial = primary "
        "value right but secondary fields wrong/missing; incorrect = primary "
        "requested value wrong or absent."
    )
    numeric_within_tolerance: Optional[bool] = Field(
        default=None,
        description="True/False if the reference contains numeric values; null "
        "if the answer is purely categorical/textual.",
    )
    matched_fields: list[str] = Field(default_factory=list)
    missing_or_wrong_fields: list[str] = Field(default_factory=list)
    rationale: str = Field(description="1-3 sentences justifying the verdict.")


JUDGE_SYSTEM = """You are a strict, fair evaluator for a cheminformatics agent.
You are given a question, a reference answer (and, when available, a structured
parsed form of it), and the agent's free-text answer. Decide whether the agent
answer matches the reference.

SCORING RULES
- Numeric values: count as a match when within +/-2% relative tolerance (or
  +/-1e-3 absolute when the reference value is ~0). The agent uses the same
  underlying RDKit/QSAR models as the reference, so correct values should be
  very close; a large deviation is WRONG, not a rounding difference.
- Categorical values (GHS hazard class, H-codes, signal word DANGER/WARNING,
  Tox21 active/inactive, AD inside/outside, alert match yes/no): must match in
  meaning, ignoring formatting/case/phrasing differences.
- SMILES: treat as correct when they denote the same molecule (canonical
  equivalence); ignore cosmetic formatting differences.
- The agent answer may include extra correct reasoning, caveats, or AD/units
  context. Do NOT penalize extra correct content. Penalize only content that is
  wrong, contradictory, or required-but-missing.

VERDICT BY answer_type
- "dict": reference_parsed lists the required fields/keys. correct = every
  required field conveyed correctly; partial = the primary requested quantity
  is right but a secondary field is wrong/missing; incorrect = the primary
  requested value is wrong or absent.
- "scalar": reference_parsed is {"value": ...}; correct iff that single value
  matches, else incorrect.
- "prose": reference_parsed is null; judge whether the agent answer conveys the
  same key conclusions/claims as the reference answer. correct = all key claims
  present and none contradicted; partial = some; incorrect = main conclusion
  wrong or absent.

OUTPUT
Populate matched_fields / missing_or_wrong_fields with the reference keys (dict/
scalar) or short claim labels (prose). Set numeric_within_tolerance to
true/false when numeric fields exist, else null. Keep rationale to 1-3 sentences.
"""


def build_judge(model: Optional[str] = 'gpt-5.4'):
    """LLM judge with structured output. Defaults to EVAL_JUDGE_MODEL env or SUMMARY_MODEL."""
    base = init_chat_model(model, model_provider="openai", api_key=OPENAI_API_KEY)
    return base.with_structured_output(JudgeVerdict)


def judge_answer(judge, row: dict, agent_answer: str) -> dict:
    """Ask the judge to grade one agent answer against the reference; return a dict."""
    payload = {
        "question": row.get("question", ""),
        "answer_type": row.get("answer_type", ""),
        "reference_answer": row.get("ref_answer", ""),
        "reference_parsed": row.get("_ref_parsed"),
        "agent_answer": agent_answer,
    }
    message = JUDGE_SYSTEM + "\n\nEVALUATION INPUT (JSON):\n" + json.dumps(
        payload, ensure_ascii=False, indent=2
    )
    try:
        verdict: JudgeVerdict = judge.invoke(message)
        return verdict.model_dump()
    except Exception as exc:
        return {
            "verdict": "error",
            "numeric_within_tolerance": None,
            "matched_fields": [],
            "missing_or_wrong_fields": [],
            "rationale": f"judge failed: {type(exc).__name__}: {exc}",
        }

## 9. Score one row (run + judge)

`score_row` is the unit of work for a single question: run the agent, then ask the
judge (unless the agent crashed). It returns **one flat record** — inputs, the agent's
answer, the judge verdict, the trace, and the `condition` tag — which is exactly what
gets saved to disk.

In [ ]:
def score_row(agent, judge, row: dict, *, condition: str = "with_skills", **run_kwargs) -> dict:
    """Run the agent on one row and score it; return a single result record."""
    run = run_agent_on_query(agent, row.get("question", ""), **run_kwargs)
    answer = run.get("answer", "")
    if run["ok"]:
        verdict = judge_answer(judge, row, answer)
    else:
        verdict = {
            "verdict": "incorrect",
            "numeric_within_tolerance": None,
            "matched_fields": [],
            "missing_or_wrong_fields": [],
            "rationale": f"agent run failed: {run.get('error')}",
        }
    return {
        "condition": condition,
        "question_id": row.get("question_id"),
        "domain": row.get("domain"),
        "difficulty": row.get("difficulty"),
        "answer_type": row.get("answer_type"),
        "question": row.get("question"),
        "ref_answer": row.get("ref_answer"),
        "ref_answer_parsed": row.get("ref_answer_parsed"),
        "agent_answer": answer,
        "agent_ok": run["ok"],
        "error": run.get("error"),
        "elapsed_s": run.get("elapsed_s"),
        "trace": run.get("trace"),
        "judge": verdict,
        "verdict": verdict.get("verdict"),
    }

## 10. Save results so runs are resumable

Results are written as **JSON Lines** (one JSON object per line) and **appended**, so a
run can be stopped and resumed without losing work. `load_results` reads them back.
`done_keys` returns the `(condition, question_id)` pairs already finished successfully —
the orchestrator uses it to skip work that's already done.

In [ ]:
def append_result(record: dict, path: Union[str, Path] = RESULTS_PATH) -> None:
    """Append one result record as a line of JSON."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def load_results(path: Union[str, Path] = RESULTS_PATH) -> list[dict]:
    """Read back all result records from a JSONL file (empty list if none)."""
    path = Path(path)
    if not path.exists():
        return []
    out = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return out


def done_ids(path: Union[str, Path] = RESULTS_PATH) -> set:
    """question_ids that already have a successful run (any condition)."""
    return {str(r.get("question_id")) for r in load_results(path) if r.get("agent_ok")}


def done_keys(path: Union[str, Path] = RESULTS_PATH) -> set:
    """(condition, question_id) pairs already finished successfully — the resume key."""
    return {
        (str(r.get("condition", "with_skills")), str(r.get("question_id")))
        for r in load_results(path)
        if r.get("agent_ok")
    }

## 11. Orchestrate a run over many rows

`run_eval` loops over rows: skip anything already done (resume), score each remaining
row, append the result, and print progress. The `condition` string tags every record
and is part of the resume key, so both conditions can safely share one results file.

`run_with_skills` / `run_no_skills` are thin wrappers that select the right agent and
condition for you (the agent's own system prompt is what makes it "with" or "no"
skills — there is nothing else to pass).

In [ ]:
def run_eval(
    rows: list[dict],
    agent=None,
    judge=None,
    *,
    condition: str = "with_skills",
    limit: Optional[int] = None,
    resume: bool = True,
    path: Union[str, Path] = RESULTS_PATH,
    verbose: bool = True,
    **run_kwargs,
) -> list[dict]:
    """Run the agent + judge over `rows`, appending each result to JSONL (resumable)."""
    agent = agent or build_eval_agent()
    judge = judge or build_judge(model = 'gpt-5.4')
    set_eval_scope()

    skip = done_keys(path) if resume else set()
    todo = [r for r in rows if (condition, str(r.get("question_id"))) not in skip]
    if limit is not None:
        todo = todo[:limit]

    if verbose:
        print(
            f"[{condition}] running {len(todo)} rows "
            f"(skipping {sum(1 for k in skip if k[0] == condition)} already done). -> {path}"
        )

    out: list[dict] = []
    for i, row in enumerate(todo, 1):
        record = score_row(agent, judge, row, condition=condition, **run_kwargs)
        append_result(record, path)
        out.append(record)
        if verbose:
            print(
                f"[{condition}][{i}/{len(todo)}] qid={record['question_id']} "
                f"{record['domain']}/{record['answer_type']} -> "
                f"{record['verdict']} ({record['elapsed_s']}s)"
            )
    return out


def run_with_skills(rows, agent=None, judge=None, **kwargs):
    """Run the with-skills condition (real execute agent: has read_files + skill prompt)."""
    return run_eval(
        rows,
        agent=agent or build_eval_agent(),
        judge=judge,
        condition="with_skills",
        **kwargs,
    )


def run_no_skills(rows, agent=None, judge=None, **kwargs):
    """Run the no-skills baseline (python_executor only + skill-free prompt)."""
    return run_eval(
        rows,
        agent=agent or build_eval_agent_no_skills(),
        judge=judge,
        condition="no_skills",
        **kwargs,
    )

## 12. Aggregate and compare the two conditions

`aggregate` turns the saved records into accuracy tables (by domain / answer_type /
difficulty), optionally for a single condition.

`compare_conditions` puts the two conditions side by side: overall accuracy, per-domain
accuracy **with the difference**, and a per-question paired view — including which
questions the skills **fixed** (`helped`) and which they **broke** (`hurt`).

(Both call `import pandas as pd`; results without a `condition` field are treated as
`with_skills`, so older single-condition runs still aggregate correctly.)

In [ ]:
def aggregate(
    results: Optional[list[dict]] = None,
    path: Union[str, Path] = RESULTS_PATH,
    *,
    condition: Optional[str] = None,
):
    """Accuracy by domain / answer_type / difficulty. Pass `condition` to filter."""
    import pandas as pd

    results = results if results is not None else load_results(path)
    if not results:
        print("No results yet.")
        return None, None

    df = pd.DataFrame(results)
    if "condition" not in df.columns:
        df["condition"] = "with_skills"
    df["condition"] = df["condition"].fillna("with_skills")
    if condition is not None:
        df = df[df["condition"] == condition]
        if df.empty:
            print(f"No results for condition={condition!r}.")
            return None, None
    df["correct"] = df["verdict"].eq("correct")
    df["correct_or_partial"] = df["verdict"].isin(["correct", "partial"])

    def _by(col):
        grp = df.groupby(col)
        return pd.DataFrame(
            {
                "n": grp.size(),
                "accuracy": grp["correct"].mean().round(3),
                "acc_incl_partial": grp["correct_or_partial"].mean().round(3),
            }
        )

    summary = {
        "overall": {
            "n": len(df),
            "accuracy": round(df["correct"].mean(), 3),
            "acc_incl_partial": round(df["correct_or_partial"].mean(), 3),
            "agent_run_failures": int((~df["agent_ok"]).sum()),
        },
        "by_domain": _by("domain"),
        "by_answer_type": _by("answer_type"),
        "by_difficulty": _by("difficulty"),
    }
    return df, summary


def compare_conditions(
    path: Union[str, Path] = RESULTS_PATH,
    *,
    conditions: tuple = ("with_skills", "no_skills"),
):
    """Side-by-side comparison of the two conditions from a shared results file.

    Returns: overall (per condition), by_domain (+delta), paired (per-question),
    helped (correct only with skills), hurt (correct only without skills).
    """
    import pandas as pd

    results = load_results(path)
    if not results:
        print("No results yet.")
        return None
    df = pd.DataFrame(results)
    if "condition" not in df.columns:
        df["condition"] = "with_skills"
    df["condition"] = df["condition"].fillna("with_skills")
    df["correct"] = df["verdict"].eq("correct")
    df["correct_or_partial"] = df["verdict"].isin(["correct", "partial"])

    overall = (
        df.groupby("condition")
        .agg(
            n=("correct", "size"),
            accuracy=("correct", "mean"),
            acc_incl_partial=("correct_or_partial", "mean"),
            run_failures=("agent_ok", lambda s: int((~s.astype(bool)).sum())),
        )
        .round(3)
    )

    by_domain = df.pivot_table(
        index="domain", columns="condition", values="correct", aggfunc="mean"
    ).round(3)
    c0, c1 = conditions
    if c0 in by_domain.columns and c1 in by_domain.columns:
        by_domain[f"delta({c0}-{c1})"] = (by_domain[c0] - by_domain[c1]).round(3)

    paired = df.pivot_table(
        index="question_id", columns="condition", values="correct", aggfunc="first"
    )
    helped, hurt = [], []
    if c0 in paired.columns and c1 in paired.columns:
        both = paired.dropna(subset=[c0, c1])
        helped = both[both[c0] & ~both[c1]].index.tolist()
        hurt = both[~both[c0] & both[c1]].index.tolist()

    return {
        "overall": overall,
        "by_domain": by_domain,
        "paired": paired,
        "helped": helped,
        "hurt": hurt,
        "df": df,
    }

## 13. Instantiate — build both agents, the judge, and load the data

Everything above only *defined* things. The cell below actually constructs the two
agents and the judge, and loads the dataset. (Constructing the agents does not call the
API; only running them in the next sections does.)

In [ ]:
set_eval_scope()

agent_skills    = build_eval_agent()   # with skills (has read_files)
agent_no_skills = build_eval_agent_no_skills()        # python_executor only
judge           = build_judge(model = 'gpt-5.4')
rows            = load_dataset('cheminformatics_eval_dataset.csv')

## 14. Smoke test — run both conditions on a few questions

A tiny end-to-end check before any big run: two cheap, pure-RDKit questions (physchem +
similarity) under **both** conditions. We deliberately avoid a Tox21 question here — the
first one trains a DeepChem model (~5 min). Output goes to a separate
`eval_runs/smoke_compare.jsonl` so it doesn't mix into the main results file.

In [ ]:
def first_of(domain):
    return next(r for r in rows if r["domain"] == domain)

smoke_rows = [first_of("physicochemical_properties"), first_of("similarity_search")]
smoke_path = RESULTS_DIR / "smoke_compare.jsonl"
if smoke_path.exists():
    smoke_path.unlink()

print(">>> WITH SKILLS")
run_with_skills(smoke_rows, agent=agent_skills, judge=judge, path=smoke_path, recursion_limit=60)
print("\n>>> NO SKILLS")
run_no_skills(smoke_rows, agent=agent_no_skills, judge=judge, path=smoke_path, recursion_limit=60)

### Inspect the smoke results (note the `skills_read` / tools difference)

In [ ]:
for rec in load_results(smoke_path):
    print("=" * 92)
    print(f"[{rec['condition']}] qid={rec['question_id']}  "
          f"[{rec['domain']}/{rec['answer_type']}]  ->  VERDICT: {rec['verdict']}")
    print("-" * 92)
    print("REF:   ", rec["ref_answer"][:160])
    print("ANSWER:", (rec["agent_answer"] or "")[:300].replace("\n", " "))
    print("skills_read:", rec["trace"]["skills_read"],
          "| skills_in_code:", rec["trace"]["skills_used_in_code"],
          "| tools:", rec["trace"]["tool_counts"])

## 15. Run the full evaluation (resumable) — both conditions

**Cost / time:** every row is a full agent loop (real OpenAI calls + RDKit / admet-ai /
DeepChem / PubChem execution). The **first** Tox21 question trains a DeepChem model
(~5 min, cached afterwards). Running all 250 questions **twice** (both conditions) is
substantial.

Both wrappers append to `eval_runs/results.jsonl` and are **resumable** — a
`(condition, question_id)` already done is skipped, so you can stop and re-run freely.
Uncomment a block below to start.

In [ ]:
# --- FULL RUN, both conditions (uncomment; resumable). ---
run_with_skills(rows, agent=agent_skills, judge=judge)
run_no_skills(rows, agent=agent_no_skills, judge=judge)

print("Uncomment a block above to run. Results ->", RESULTS_PATH)

## 16. Results per condition

In [ ]:
for cond in ("with_skills", "no_skills"):
    df_c, summary_c = aggregate(condition=cond)   # reads eval_runs/results.jsonl
    if summary_c:
        print(f"\n===== {cond} =====")
        print("OVERALL:", summary_c["overall"])
        display(summary_c["by_domain"])

## 17. Compare: with skills vs no skills

`compare_conditions` reads the shared results file and shows overall accuracy per
condition, per-domain accuracy with the delta, the paired per-question view, and the
`helped` / `hurt` lists. If the full run hasn't been done yet, it falls back to the
smoke comparison so the cell still demonstrates the output.

In [ ]:
cmp = compare_conditions()  # reads eval_runs/results.jsonl
if cmp is None:
    print("No full-run results yet — run section 15 first. Showing the smoke comparison:")
    cmp = compare_conditions(path=RESULTS_DIR / "smoke_compare.jsonl")

if cmp:
    print("OVERALL (accuracy by condition):")
    display(cmp["overall"])
    print("\nBY DOMAIN (accuracy per condition + delta):")
    display(cmp["by_domain"])
    print("\nPAIRED per-question correctness (qids run in both):")
    display(cmp["paired"])
    print(f"\nSkills FIXED (correct only with skills): {cmp['helped']}")
    print(f"Skills BROKE (correct only without skills): {cmp['hurt']}")